# BufferWise — Your Intelligent Day Planner

**An autonomous multi-agent scheduling assistant**

BufferWise helps a user create a realistic day plan. It checks meeting gaps, estimates travel time, inserts preparation and rest buffers, adapts to traffic and (optionally) live weather, and explains every decision.

Traffic is simulated by a slider, weather can be fetched from Open-Meteo

## 1. Install the UI and supporting libraries

Gradio creates the web interface inside this notebook. Pandas handles the meeting table, and Requests is used only when the optional live-weather lookup is enabled.

Run this cell once when you open the notebook in Google Colab.


In [16]:
!pip -q install gradio pandas requests

## 2. Imports

An intelligent agent begins with **knowledge about its environment**.

Here, BufferWise knows a few example locations and approximate travel times. These are editable demonstration values, not live map directions. The agent also has baseline rules: 15 minutes for preparation before a high-stakes meeting, and a 10-minute break after a run of tightly spaced meetings.

The `get_weather_adjustment()` function optionally asks the free Open-Meteo service for current weather. It converts the API response into simple conditions the agent understands, such as Rain, Fog, Heat or StrongWind. If the API is unavailable, the planner can still run using its baseline settings.


In [17]:
import requests
import pandas as pd
import math

from datetime import datetime, timedelta
from math import ceil

In [6]:

import gradio as gr
import pandas as pd
import requests
import math
import json
import hmac
import hashlib
import time
import secrets

from datetime import datetime, timedelta
from math import ceil

APP_NAME = "BufferWise — Your Intelligent Day Planner"


# --------------------------------------------------
# DISTANCE-BASED TRAVEL TIME CALCULATOR
# --------------------------------------------------

# Assumed average speeds in km/h.
# These are illustrative values, not live traffic measurements.
AVERAGE_SPEED_KMPH = {
    "Car": 30,
    "Bike": 25,
    "Public transport": 20
}


def estimate_travel(
    origin,
    destination,
    distance_km=None,
    travel_mode="Car",
    traffic_multiplier=1.0,
    manual_minutes=None
):
    """
    Estimate travel time using distance and average speed.

    Manual travel time takes priority when provided.
    Returns None if the distance or manual duration is missing.
    """

    # No travel needed for online meetings or the same location.
    if origin == destination or "Online" in (origin, destination):
        return 0

    # Use manually entered travel time if available.
    if manual_minutes is not None and not pd.isna(manual_minutes):
        base_time = float(manual_minutes)

    # Otherwise calculate travel time from distance.
    elif distance_km is not None and not pd.isna(distance_km):
        distance_km = float(distance_km)

        if distance_km < 0:
            raise ValueError("Distance cannot be negative.")

        average_speed = AVERAGE_SPEED_KMPH.get(travel_mode)

        if average_speed is None:
            raise ValueError("Invalid travel mode.")

        base_time = (distance_km / average_speed) * 60

    else:
        # Do not invent a travel-time estimate.
        return None

    if base_time < 0:
        raise ValueError("Travel time cannot be negative.")

    # Apply traffic or weather multiplier.
    adjusted_time = base_time * max(1.0, float(traffic_multiplier))

    # Round up to the next whole minute.
    return ceil(adjusted_time)


# --------------------------------------------------
# WEATHER INFORMATION
# --------------------------------------------------

def get_weather_adjustment(city):
    """Fetch current weather and translate it into agent-readable conditions."""

    if not city.strip():
        return {
            "conditions": [],
            "travel_multiplier": 1.0,
            "extra_buffer": 0,
            "extra_break": 0,
            "message": "No city entered; using baseline weather."
        }

    try:
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_response = requests.get(
            geo_url,
            params={"name": city, "count": 1},
            timeout=8
        )
        geo_response.raise_for_status()
        geo = geo_response.json()

        if not geo.get("results"):
            raise ValueError("City not found")

        place = geo["results"][0]

        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": (
                    "temperature_2m,apparent_temperature,"
                    "weather_code,wind_speed_10m,precipitation"
                ),
                "timezone": "auto"
            },
            timeout=8
        )

        weather_response.raise_for_status()
        data = weather_response.json()["current"]

        code = int(data.get("weather_code", 0))
        feels = float(
            data.get("apparent_temperature",
                     data.get("temperature_2m", 25))
        )
        wind = float(data.get("wind_speed_10m", 0))

        conditions = []
        multiplier = 1.0
        extra_buffer = 0
        extra_break = 0

        if code in [45, 48]:
            conditions.append("Fog")
            multiplier = max(multiplier, 1.30)
            extra_buffer += 10

        elif code in [51, 53, 55, 56, 57, 61, 63, 65,
                      66, 67, 80, 81, 82]:
            conditions.append("Rain")
            multiplier = max(multiplier, 1.25)
            extra_buffer += 10

        elif code >= 95:
            conditions.append("Thunderstorm")
            multiplier = max(multiplier, 1.50)
            extra_buffer += 15

        if feels >= 38:
            conditions.append("Heat")
            extra_break += 5

        if wind >= 40:
            conditions.append("StrongWind")
            multiplier = max(multiplier, 1.10)

        if not conditions:
            conditions.append("Clear / normal")

        description = (
            f"Weather in {place.get('name', city)}: "
            f"{', '.join(conditions)}. "
            f"Feels like {feels:.1f}°C; "
            f"wind {wind:.1f} km/h. "
            f"Travel multiplier ×{multiplier:.2f}."
        )

        return {
            "conditions": conditions,
            "travel_multiplier": multiplier,
            "extra_buffer": extra_buffer,
            "extra_break": extra_break,
            "message": description
        }

    except Exception as exc:
        return {
            "conditions": [],
            "travel_multiplier": 1.0,
            "extra_buffer": 0,
            "extra_break": 0,
            "message": (
                f"Weather lookup unavailable ({exc}). "
                "Using baseline weather."
            )
        }


print("BufferWise core knowledge loaded.")
print("Distance-based travel calculator loaded.")


BufferWise core knowledge loaded.
Distance-based travel calculator loaded.


## 3. The scheduling agent: perceive → reason → act

This is the main agent loop.

For every meeting, BufferWise:
1. **Perceives** the meeting time, location, importance and the gap after the previous meeting's adjusted end.
2. **Reasons** using prioritised condition–action rules: conflict, travel buffer, preparation buffer, biological break, or schedule OK.
3. **Acts** by shifting a meeting when a buffer is needed.
4. **Carries forward** the adjusted end time, so one change can affect later meetings.

Traffic estimation can operate automatically using a predefined traffic assumption, or manually when the user selects Normal, Moderate, or Heavy traffic. Weather information can provide an additional travel-time multiplier when live weather is enabled. from Weather can add another travel-time multiplier. The BDI summary represents the agent's beliefs, desires and current intention: arrive on time, notify the user, and keep the schedule practical.


In [11]:

def to_minutes(clock_text):
    h, m = map(int, str(clock_text).strip().split(":"))
    return h * 60 + m


def to_clock(minutes):
    minutes = int(round(minutes))
    return f"{(minutes // 60) % 24:02d}:{minutes % 60:02d}"


def run_bufferwise(
    meetings,
    city="",
    use_live_weather=False,
    travel_mode="Car",
    safety_buffer=5,
    traffic_mode="Automatic",
    traffic_level="Normal"
):
    """Run one readable BufferWise planning cycle."""

    if meetings is None or len(meetings) == 0:
        return pd.DataFrame(), "Please add at least one meeting.", "No agent cycle was run."

    # Determine traffic multiplier.
    traffic_levels = {
        "Normal": 1.0,
        "Moderate": 1.2,
        "Heavy": 1.5
    }

    if traffic_mode == "Automatic":
        traffic_multiplier = 1.2
    else:
        traffic_multiplier = float(traffic_level)

    # Get weather adjustment if enabled.
    weather = get_weather_adjustment(city) if use_live_weather else {
        "conditions": ["Simulated / not fetched"],
        "travel_multiplier": 1.0,
        "extra_buffer": 0,
        "extra_break": 0,
        "message": "Live weather is off; using baseline weather."
    }

    records = []
    warnings = []

    # Convert the Gradio table into clean meeting records.
    for _, row in meetings.iterrows():

        title = str(row.get("Meeting", "")).strip()

        if not title or title.lower() == "nan":
            continue

        start = to_minutes(row["Start"])
        end = to_minutes(row["End"])

        if end <= start:
            raise ValueError(
                f"End time must be after start time for '{title}'."
            )

        location = str(row.get("Location", "Office")).strip()

        high = str(row.get("High stakes", "No")).strip().lower() in [
            "yes", "true", "1"
        ]

        # Read distance from the previous meeting.
        distance = row.get("Distance from previous (km)", None)

        if distance is not None and pd.isna(distance):
            distance = None

        # Read optional manually entered travel duration.
        manual_travel = row.get("Manual travel (min)", None)

        if manual_travel is not None and pd.isna(manual_travel):
            manual_travel = None

        if distance is not None and float(distance) < 0:
            raise ValueError(
                f"Distance cannot be negative for '{title}'."
            )

        if manual_travel is not None and float(manual_travel) < 0:
            raise ValueError(
                f"Manual travel time cannot be negative for '{title}'."
            )

        records.append({
            "title": title,
            "start": start,
            "end": end,
            "location": location,
            "high": high,
            "distance": distance,
            "manual_travel": manual_travel
        })

    records.sort(key=lambda x: x["start"])

    if not records:
        return pd.DataFrame(), "No valid meetings found.", "Add meeting names and times."

    output = []
    logs = []

    previous = None
    previous_effective_end = None

    tight_streak = 0
    conflicts = 0
    inserted = 0
    travel_total = 0

    for index, meeting in enumerate(records):

        original_start = meeting["start"]
        original_end = meeting["end"]

        new_start = original_start
        new_end = original_end

        action = "Schedule OK"
        reason = "No rule requires a change."

        travel_need = 0
        gap = None

        # Check the transition from the previous meeting.
        if previous is not None:

            gap = original_start - previous_effective_end

            # Estimate travel only when the locations differ.
            travel_need = estimate_travel(
                previous["location"],
                meeting["location"],
                distance_km=meeting["distance"],
                travel_mode=travel_mode,
                traffic_multiplier=(
                    traffic_multiplier *
                    weather["travel_multiplier"]
                ),
                manual_minutes=meeting["manual_travel"]
            )

            # Missing distance means the agent cannot safely
            # calculate travel time for this transition.
            if travel_need is None:
                warnings.append(
                    f"Travel distance missing for transition "
                    f"{previous['location']} → {meeting['location']} "
                    f"before '{meeting['title']}'."
                )

                logs.append(
                    f"• **{meeting['title']}** — Travel estimate unavailable. "
                    f"Enter the distance or manual travel time."
                )

                # Prevent the agent from silently assuming zero travel.
                travel_need = None

            else:
                travel_total += travel_need

            tight_streak = tight_streak + 1 if gap < 15 else 0

            # Rule 1: resolve overlap first.
            if gap < 0:

                shift = previous_effective_end - original_start

                new_start += shift
                new_end += shift

                action = "Conflict resolved"

                reason = (
                    f"Overlapped the previous meeting by "
                    f"{abs(gap)} min; moved after it."
                )

                conflicts += 1
                inserted += shift

            # Rule 2: provide the required travel time.
            elif travel_need is None:

                action = "Travel information required"

                reason = (
                    "The locations differ, but no distance or manual "
                    "travel duration was provided. Travel time cannot "
                    "be estimated reliably."
                )

            elif gap < travel_need + safety_buffer:

                required_time = travel_need + safety_buffer

                shift = required_time - gap

                new_start += shift
                new_end += shift

                action = "Travel buffer"

                reason = (
                    f"Only {gap} min available; estimated travel "
                    f"needs {travel_need} min plus "
                    f"{safety_buffer} min safety buffer."
                )

                inserted += shift

            # Rule 3: prepare before an important meeting.
            elif meeting["high"] and gap < 15 + weather["extra_buffer"]:

                shift = 15 + weather["extra_buffer"]

                new_start += shift
                new_end += shift

                action = "Preparation buffer"

                reason = (
                    f"High-stakes meeting: {shift} min "
                    f"preparation buffer added."
                )

                inserted += shift

            # Rule 4: take a break after several tightly spaced meetings.
            elif tight_streak >= 3 and gap < 10 + weather["extra_break"]:

                shift = 10 + weather["extra_break"]

                new_start += shift
                new_end += shift

                action = "Biological break"

                reason = (
                    f"{tight_streak} tight transitions detected; "
                    f"{shift} min break added."
                )

                inserted += shift

        output.append({
            "Meeting": meeting["title"],
            "Original time": (
                f"{to_clock(original_start)}–{to_clock(original_end)}"
            ),
            "Updated time": (
                f"{to_clock(new_start)}–{to_clock(new_end)}"
            ),
            "Location": meeting["location"],
            "Distance (km)": (
                meeting["distance"]
                if meeting["distance"] is not None
                else "Not provided"
            ),
            "Travel estimate (min)": (
                travel_need
                if travel_need is not None
                else "Unknown"
            ),
            "Agent action": action
        })

        logs.append(
            f"• **{meeting['title']}** — {action}. {reason}"
            + (
                f" (Gap: {gap} min; travel: "
                f"{travel_need} min.)"
                if gap is not None and travel_need is not None
                else ""
            )
        )

        previous = meeting
        previous_effective_end = new_end

    # Planning summary.
    summary = (
        f"### Planning result\n"
        f"- Meetings reviewed: **{len(output)}**\n"
        f"- Overlaps resolved: **{conflicts}**\n"
        f"- Total minutes shifted/added: **{inserted}**\n"
        f"- Estimated travel time: **{travel_total} min**\n"
        f"- Travel mode: **{travel_mode}**\n"
        f"- Safety buffer: **{safety_buffer} min**\n"
        f"- Traffic estimation: **{traffic_mode}**\n"
        f"- Traffic condition: **{'Default assumption' if traffic_mode == 'Automatic' else f'Manual setting (×{traffic_multiplier:.2f})'}**\n"
        f"- Applied traffic multiplier: **×{traffic_multiplier:.2f}**\n"
        f"- Weather: {weather['message']}\n"
    )

    if warnings:
        summary += (
            "\n### Travel information required\n"
            + "\n".join(f"- {warning}" for warning in warnings)
        )

    # BDI: Beliefs, Desires and Intentions.
    bdi = (
        "### Agent's current BDI state\n"
        "**Beliefs**\n"
        f"- Calendar contains {len(output)} meetings.\n"
        f"- Traffic estimation mode is {traffic_mode}.\n"
        f"- Traffic multiplier is ×{traffic_multiplier:.2f}.\n"
        f"- Travel mode is {travel_mode}.\n"
        f"- Active weather conditions: "
        f"{', '.join(weather['conditions'])}.\n\n"

        "**Desires**\n"
        "- ArriveOnTime: ACTIVE\n"
        "- NotifyUser: ACTIVE\n"
        "- KeepSchedulePractical: ACTIVE\n\n"

        "**Intention**\n"
        "- Review meetings in order → calculate travel "
        "requirements → check each gap → apply the highest-priority "
        "rule → update the next meeting.\n"
        "- Final state: PLAN GENERATED."
    )

    decision_trace = (
        bdi
        + "\n\n### Decision trace\n"
        + "\n".join(logs)
    )

    return pd.DataFrame(output), summary, decision_trace


## 4. Cooperation and Security: How the Wider Agent System Works

The original project goes beyond a single scheduling planner and includes cooperation and secure communication between agents:

- **Multi-agent negotiation:** Two agents exchange proposals and counter-proposals until they find and confirm a mutually acceptable meeting slot.
- **Socket messaging:** Agents exchange typed JSON messages over TCP. The communication protocol uses a 4-byte length header to identify message boundaries and ensure that complete messages are received.
- **Authentication and integrity:** Before acting on a received message, the receiver checks its format, trusted sender identity, HMAC-SHA256 signature, timestamp freshness, nonce/replay protection, and sender permissions.

The small demonstration below illustrates the security concept locally. It signs a command, verifies its authenticity, and then demonstrates that modifying the payload causes verification to fail.

**Scope note:** This demonstration runs locally. It does not establish a TCP socket connection or replace the original secure server and full communication protocol.

In [12]:

def security_demo():
    # Shared secret key for this local demonstration.
    secret_key = b"bufferwise-demo-secret"

    # Create a sample message from Agent A.
    signed_fields = {
        "sender": "agent-a",
        "type": "STATE_UPDATE",
        "payload": {"meeting_count": 4},
        "nonce": secrets.token_hex(8),
        "ts": time.time()
    }

    # Generate an HMAC-SHA256 signature for a message.
    def make_signature(message):
        canonical = json.dumps(
            message,
            sort_keys=True,
            separators=(",", ":")
        ).encode()

        return hmac.new(
            secret_key,
            canonical,
            hashlib.sha256
        ).hexdigest()

    # Sign the original message.
    signature = make_signature(signed_fields)

    # Receiver verifies the original message.
    received_signature = make_signature(signed_fields)

    valid = hmac.compare_digest(
        signature,
        received_signature
    )

    # Simulate an attacker changing the message payload.
    changed_message = dict(signed_fields)
    changed_message["payload"] = {"meeting_count": 99}

    # Verify the modified message using the original signature.
    changed_signature = make_signature(changed_message)

    changed = hmac.compare_digest(
        signature,
        changed_signature
    )

    return (
        "### Local message verification demo\n"
        f"- Trusted sender: **agent-a**\n"
        f"- Command: **{signed_fields['type']}**\n"
        f"- Original message signature: **{'VALID' if valid else 'REJECTED'}**\n"
        f"- Modified payload signature: **{'VALID' if changed else 'REJECTED'}**\n\n"
        "The original message is accepted because its signature matches. "
        "The modified payload is rejected because changing the message "
        "changes its HMAC signature.\n\n"
        "**Security scope:** This local demonstration illustrates "
        "message integrity and signature verification. The full project "
        "receiver additionally checks timestamp freshness, nonce reuse, "
        "trusted sender identity and command permissions. "
        "This demo does not establish a TCP connection."
    )


def negotiation_demo(alice_preference, bob_preference):

    # Available meeting slots.
    slots = [
        "10:00",
        "11:00",
        "12:00",
        "13:00",
        "14:00",
        "15:00"
    ]

    # Calculate how suitable a slot is for an agent.
    # A slot closer to the preferred time gets a higher score.
    def score(slot, preferred):
        distance = abs(
            to_minutes(slot) - to_minutes(preferred)
        )

        return max(
            0,
            round(1 - distance / 300, 3)
        )

    ranked = []

    # Calculate both agents' preferences for every slot.
    for slot in slots:

        alice_score = score(slot, alice_preference)
        bob_score = score(slot, bob_preference)

        # Calculate the average utility of both agents.
        combined_score = round(
            (alice_score + bob_score) / 2,
            3
        )

        ranked.append((
            combined_score,
            slot,
            alice_score,
            bob_score
        ))

    # Highest combined score first.
    # If scores tie, prefer the earlier time.
    ranked.sort(
        key=lambda item: (
            item[0],
            -to_minutes(item[1])
        ),
        reverse=True
    )

    best = ranked[0]

    return (
        "### Illustrative negotiation result\n"
        f"- Alice's preferred time: **{alice_preference}** "
        f"(utility {best[2]:.3f} for selected slot)\n"
        f"- Bob's preferred time: **{bob_preference}** "
        f"(utility {best[3]:.3f} for selected slot)\n"
        f"- Best shared slot in this simple demo: **{best[1]}**\n\n"
        "The assignment's full negotiation uses alternating offers, "
        "aspiration/concession thresholds, reservation values and an "
        "ACCEPT → CONFIRM → FINAL_ACK handshake. "
        "This compact UI example only illustrates how agents can "
        "compare and rank shared meeting slots."
    )


## 5. Gradio Interface

The Gradio interface is organised around the user's tasks rather than individual assignment numbers. It provides three main sections:

- **Plan My Day:** Enter or edit meetings, select the travel mode, configure traffic estimation and safety buffers, optionally fetch live weather information, and run the BufferWise scheduling agent.
- **Agent's Reasoning:** View the agent's BDI (Beliefs, Desires and Intentions) state and decision trace to understand why particular scheduling decisions or meeting adjustments were made.
- **Coordination & Trust:** Explore an illustrative shared-slot negotiation between two agents and a local message-integrity demonstration using HMAC-SHA256.

The interface comes with a pre-filled sample calendar so users can immediately explore the planner's functionality. They can modify the existing entries or add their own meetings to test different scheduling scenarios.

**Note:** The coordination and security sections provide compact demonstrations of the concepts; they do not replace the complete TCP communication, multi-agent negotiation, or secure server implementations.

In [13]:

import re
from datetime import datetime, timedelta
import pandas as pd
import gradio as gr

PLACES = ["Office", "Downtown", "Client Site", "Home", "Online"]


def guess_place(location, title=""):
    text = (str(location) + " " + str(title)).lower()

    if any(x in text for x in ["zoom", "teams", "online", "meet.google", "http"]):
        return "Online"
    if "home" in text:
        return "Home"
    if "client" in text:
        return "Client Site"
    if any(x in text for x in ["downtown", "hq"]):
        return "Downtown"

    return "Office"


def parse_csv_schedule(path):
    df = pd.read_csv(path)
    columns = {c.lower().strip(): c for c in df.columns}

    def find_column(*names):
        return next((columns[n] for n in names if n in columns), None)

    title_col = find_column("meeting", "title", "summary", "subject")
    start_col = find_column("start", "start time", "from")
    end_col = find_column("end", "end time", "to")
    place_col = find_column("location", "place")
    distance_col = find_column(
        "distance from previous (km)",
        "distance (km)",
        "distance"
    )
    manual_travel_col = find_column(
        "manual travel (min)",
        "travel time (min)",
        "manual travel"
    )
    important_col = find_column("important", "high stakes", "priority")

    if not start_col or not end_col:
        raise ValueError("CSV must contain Start and End columns.")

    rows = []

    for _, row in df.iterrows():
        try:
            title = str(row[title_col]) if title_col else "Meeting"

            start = pd.to_datetime(row[start_col]).strftime("%H:%M")
            end = pd.to_datetime(row[end_col]).strftime("%H:%M")

            location = str(row[place_col]) if place_col else ""

            # Read optional distance information.
            distance = (
                row[distance_col]
                if distance_col else None
            )

            if pd.isna(distance):
                distance = None

            # Read optional manual travel duration.
            manual_travel = (
                row[manual_travel_col]
                if manual_travel_col else None
            )

            if pd.isna(manual_travel):
                manual_travel = None

            important = (
                str(row[important_col]).lower()
                in ["yes", "true", "1", "high"]
                if important_col else False
            )

            rows.append([
                title,
                start,
                end,
                guess_place(location, title),
                distance,
                manual_travel,
                "Yes" if important else "No"
            ])

        except Exception:
            continue

    return rows


def parse_ics_schedule(path):
    with open(path, encoding="utf-8", errors="ignore") as file:
        text = file.read()

    text = re.sub(r"\r?\n[ \t]", "", text)

    rows = []

    for event in re.findall(r"BEGIN:VEVENT(.*?)END:VEVENT", text, re.S):

        fields = {}

        for line in event.splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                fields[key.split(";")[0].upper()] = value.strip()

        try:
            start_text = fields["DTSTART"]
            end_text = fields["DTEND"]

            start = datetime.strptime(
                start_text.rstrip("Z"), "%Y%m%dT%H%M%S"
            )

            end = datetime.strptime(
                end_text.rstrip("Z"), "%Y%m%dT%H%M%S"
            )

            # Skip all-day events and invalid durations.
            if end <= start:
                continue

            title = fields.get("SUMMARY", "Meeting")
            location = fields.get("LOCATION", "")

            important = any(
                word in title.lower()
                for word in [
                    "interview",
                    "pitch",
                    "presentation",
                    "exam",
                    "demo"
                ]
            )

            rows.append([
                title,
                start.strftime("%H:%M"),
                end.strftime("%H:%M"),
                guess_place(location, title),
                None,  # Distance must be entered by the user.
                None,  # Manual travel time is optional.
                "Yes" if important else "No"
            ])

        except Exception:
            continue

    return sorted(rows, key=lambda x: x[1])


def import_schedule(file):

    if file is None:
        return gr.update(), "Please choose a calendar file first."

    path = file.name if hasattr(file, "name") else str(file)

    try:

        if path.lower().endswith(".csv"):
            rows = parse_csv_schedule(path)

        elif path.lower().endswith(".ics"):
            rows = parse_ics_schedule(path)

        else:
            return gr.update(), "Please upload an .ics or .csv file."

        if not rows:
            return gr.update(), "No valid timed meetings were found."

        return (
            rows,
            f"✅ Successfully imported **{len(rows)} meetings**. "
            "Review the table and enter missing travel distances before planning."
        )

    except Exception as error:
        return gr.update(), f"Could not import the file: {error}"


In [15]:

# --------------------------------------------------
# SAMPLE CALENDAR
# --------------------------------------------------

sample_meetings = pd.DataFrame([
    ["Team Standup", "09:00", "09:30", "Office", 0, None, "No"],
    ["Client Presentation", "09:40", "10:30", "Downtown", 4, None, "Yes"],
    ["1:1 with Manager", "10:50", "11:35", "Office", 4, None, "No"],
    ["Budget Review", "11:40", "12:20", "Office", 0, None, "No"],
    ["Investor Pitch", "12:30", "13:10", "Client Site", 6, None, "Yes"],
    ["Remote Team Sync", "13:15", "13:45", "Online", 0, None, "No"],
], columns=[
    "Meeting", "Start", "End", "Location",
    "Distance from previous (km)", "Manual travel (min)", "High stakes"
])


# --------------------------------------------------
# GRADIO CALLBACK FOR BUFFERWISE
# --------------------------------------------------

def run_bufferwise_from_ui(
    meetings,
    city,
    live_weather,
    travel_mode,
    safety_buffer,
    traffic_mode,
    manual_traffic
):
    if traffic_mode == "Automatic":
        traffic_level = 1.2
    else:
        traffic_level = manual_traffic

    return run_bufferwise(
        meetings=meetings,
        city=city,
        use_live_weather=live_weather,
        travel_mode=travel_mode,
        safety_buffer=safety_buffer,
        traffic_mode=traffic_mode,
        traffic_level=traffic_level
    )


# --------------------------------------------------
# GRADIO INTERFACE
# --------------------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="BufferWise | Intelligent Day Planner"
) as demo:

    gr.Markdown(
        """
        # 🧠 BufferWise
        ### Your Intelligent Day Planner

        Your personal scheduling agent that protects your time,
        plans meeting buffers and explains every decision.
        """
    )

    # ==================================================
    # TAB 1: PLAN MY DAY
    # ==================================================

    with gr.Tab("📅 Plan my day"):

        gr.Markdown("### Import your calendar")

        gr.Markdown(
            "Upload your calendar or start with a sample schedule. "
            "You can edit the meetings before running the agent."
        )

        with gr.Row():

            calendar_file = gr.File(
                label="📥 Upload calendar (.ics or .csv)",
                file_types=[".ics", ".csv"],
                type="filepath"
            )

            with gr.Column():

                import_button = gr.Button(
                    "Import schedule",
                    variant="primary"
                )

                sample_button = gr.Button(
                    "Load sample schedule"
                )

        import_status = gr.Markdown()

        gr.Markdown("---")
        gr.Markdown("### 🗓️ Your calendar")

        meetings_input = gr.Dataframe(
            value=sample_meetings,
            headers=[
                "Meeting",
                "Start",
                "End",
                "Location",
                "Distance from previous (km)",
                "Manual travel (min)",
                "High stakes"
            ],
            datatype=[
                "str",
                "str",
                "str",
                "str",
                "number",
                "number",
                "str"
            ],
            row_count=(6, "dynamic"),
            col_count=(7, "fixed"),
            interactive=True,
            label="Editable meeting schedule"
        )

        gr.Markdown(
            """
            💡 **Calendar tips**

            - Use HH:MM time format.
            - Available locations: Office, Downtown, Client Site, Home and Online.
            - Enter the distance from the previous meeting for travel estimation.
            - Use Manual travel (min) when you already know the travel duration.
            - The first meeting does not need a travel distance.
            """
        )

        gr.Markdown("---")
        gr.Markdown("### ⚙️ Planning preferences")

        # Travel mode
        travel_mode = gr.Dropdown(
            choices=[
                "Car",
                "Bike",
                "Public transport"
            ],
            value="Car",
            label="🚗 Travel mode"
        )

        # Safety buffer
        safety_buffer = gr.Slider(
            minimum=0,
            maximum=30,
            value=5,
            step=1,
            label="⏱️ Safety buffer (minutes)"
        )

        gr.Markdown("### 🚦 Traffic estimation")

        # Choose automatic or manual traffic estimation.
        traffic_mode = gr.Radio(
            choices=[
                "Automatic",
                "Manual"
            ],
            value="Automatic",
            label="Traffic estimation mode"
        )

        # Manual slider appears only when Manual is selected.
        manual_traffic = gr.Slider(
            minimum=1.0,
            maximum=2.2,
            value=1.2,
            step=0.1,
            label="Manual traffic multiplier",
            info="1.0 = Normal, 1.2 = Moderate, 1.5 = Heavy",
            visible=False
        )

        def show_manual_traffic(mode):
            return gr.update(
                visible=(mode == "Manual")
            )

        traffic_mode.change(
            fn=show_manual_traffic,
            inputs=traffic_mode,
            outputs=manual_traffic
        )

        gr.Markdown(
            """
            **How traffic estimation works**

            - Automatic: Uses a predefined multiplier of ×1.2.
            - Manual: Adjust the multiplier using the slider.
            - The slider changes the estimated travel duration.

            Automatic mode is a default assumption, not live traffic detection.
            """
        )

        gr.Markdown("### 🌤️ Weather settings")

        with gr.Row():

            city = gr.Textbox(
                value="Chennai",
                label="City for weather"
            )

            live_weather = gr.Checkbox(
                value=False,
                label="Use current weather from Open-Meteo"
            )

        run_button = gr.Button(
            "✨ Run BufferWise Agent",
            variant="primary",
            size="lg"
        )

        gr.Markdown("### 📊 Your optimized day")

        schedule_output = gr.Dataframe(
            label="Updated meeting schedule",
            interactive=False
        )

        summary_output = gr.Markdown()

    # ==================================================
    # TAB 2: AGENT'S REASONING
    # ==================================================

    with gr.Tab("🧩 Agent's reasoning"):

        gr.Markdown(
            """
            ### How BufferWise thinks

            The agent follows a **Perceive → Reason → Act → Reconsider**
            cycle.

            - **Beliefs:** What the agent knows about your schedule.
            - **Desires:** What it wants to achieve.
            - **Intentions:** The actions it decides to take.
            """
        )

        gr.Markdown(
            """
            **Decision priority**

            1. Resolve meeting overlaps.
            2. Allow sufficient travel time.
            3. Add preparation time for important meetings.
            4. Protect breaks between consecutive meetings.
            5. Keep the schedule unchanged when no adjustment is needed.
            """
        )

        reasoning_box = gr.Markdown(
            "Run BufferWise to see its BDI state and decision trace here."
        )

        gr.Markdown(
            """
            ### Components behind the agent

            - **Rule reasoner:** Evaluates meeting gaps and applies scheduling rules.
            - **Traffic monitor:** Supplies traffic conditions in the background.
            - **Utility optimiser:** Compares possible scheduling choices.
            - **BDI interpreter:** Tracks goals and reconsiders plans.
            - **Knowledge + API:** Uses environmental information such as weather.
            - **Security gate:** Checks whether incoming agent messages can be trusted.
            """
        )

    # ==================================================
    # TAB 3: COORDINATION AND TRUST
    # ==================================================

    with gr.Tab("🤝 Coordination & trust"):

        gr.Markdown("### Multi-agent meeting coordination")

        gr.Markdown(
            "Two calendar agents can compare their preferred meeting times "
            "and find a mutually suitable slot."
        )

        with gr.Row():

            alice = gr.Dropdown(
                ["10:00", "11:00", "12:00", "13:00", "14:00"],
                value="10:00",
                label="Alice's preferred time"
            )

            bob = gr.Dropdown(
                ["10:00", "11:00", "12:00", "13:00", "14:00"],
                value="14:00",
                label="Bob's preferred time"
            )

        negotiate_btn = gr.Button("Find a shared slot")

        negotiation_result = gr.Markdown()

        negotiate_btn.click(
            negotiation_demo,
            inputs=[alice, bob],
            outputs=negotiation_result
        )

        gr.Markdown("---")
        gr.Markdown("### 🔐 Secure agent communication")

        gr.Markdown(
            "The receiver checks the message signature to detect "
            "whether its contents have been modified."
        )

        security_btn = gr.Button(
            "Run message integrity demo"
        )

        security_result = gr.Markdown()

        security_btn.click(
            security_demo,
            outputs=security_result
        )

    # ==================================================
    # FOOTER
    # ==================================================

    gr.Markdown(
        """
        ---
        <center>🧠 BufferWise · Sense clearly. Plan intelligently. Protect your time.</center>
        """
    )

    # ==================================================
    # CALENDAR IMPORT AND SAMPLE SCHEDULE
    # ==================================================

    import_button.click(
        import_schedule,
        inputs=calendar_file,
        outputs=[meetings_input, import_status]
    )

    sample_button.click(
        lambda: (
            sample_meetings,
            "✅ Sample schedule loaded. You can edit it before planning."
        ),
        outputs=[meetings_input, import_status]
    )

    # ==================================================
    # CONNECT THE BUFFERWISE AGENT
    # ==================================================

    run_button.click(
        fn=run_bufferwise_from_ui,
        inputs=[
            meetings_input,
            city,
            live_weather,
            travel_mode,
            safety_buffer,
            traffic_mode,
            manual_traffic
        ],
        outputs=[
            schedule_output,
            summary_output,
            reasoning_box
        ]
    )


demo.launch(share=True, debug=False)


/tmp/ipykernel_2819/845222916.py:51: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(
/usr/local/lib/python3.13/dist-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1bd1b68c36a636486e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
